# ARCHS4 CLAMP models with C2CP prior at 75% sample coverage (Random Sampling)

**Environment:** `clamp-analyses`  

This notebook builds 3 CLAMP models using the C2CP (C2 Canonical Pathways) prior at 75% sample coverage, each with a different random seed for sample shuffling.

Steps (repeated for each seed):
1. Subsample the preprocessed ARCHS4 data with random sample selection
2. Compute SVD on the subsampled data
3. Run CLAMPbase
4. Run CLAMPfull with C2CP prior

Each run uses a different seed (`base_seed + 0:2`) for reproducibility, allowing comparison of model stability across different sample selections.

## Load libraries

In [ ]:
library(bigstatsr)
library(data.table)
library(dplyr)
library(rsvd)
library(glmnet)
library(Matrix)
library(knitr)
library(here)
library(CLAMP)
library(PCAtools)
library(rhdf5)

source(here("config.R"))

## Configuration

In [ ]:
# Base output directory
base_output_dir <- config$ARCHS4$DATASET_FOLDER

# Coverage level for this notebook
coverage <- 0.75
coverage_pct <- coverage * 100

# Data path for priors
data_path <- here::here('data/archs4')
archs4_file <- here('data/archs4/human_gene_v2.5.h5')

# SVD parameters
N_CORES <- config$ARCHS4$CLAMP_PARAMS$RANDOM_SVD_N_CORES
MULTIPLIER <- 100
MAX_ITER   <- 5000

base_seed <- config$ARCHS4$CLAMP_PARAMS$RANDOM_SVD_SEED
seeds <- base_seed + 0:2
n_runs <- length(seeds)
message("Will run ", n_runs, " models with seeds: ", paste(seeds, collapse = ", "))

## Load preprocessed data

In [ ]:
# Load metadata
meta <- readRDS(file.path(base_output_dir, "metadata_filtered.rds"))
n_genes_thin <- meta$n_genes_thin
n_samples_total <- meta$n_samples
archs4_genes <- meta$gene_symbols_thin

# Load sample names
all_samples <- readRDS(file.path(base_output_dir, "all_samples.rds"))
sample_names_total <- all_samples[seq_len(n_samples_total)]

# Load C2CP pathway
c2_gmt <- CLAMP:::read_gmt(file.path(data_path, "c2.cp.v2026.1.Hs.symbols.gmt"))
names(c2_gmt) <- paste0("C2CP_", names(c2_gmt))
c2_pathMat <- gmtListToSparseMat(list(C2CP = c2_gmt))
C2CP_matched <- getMatchedPathwayMat(c2_pathMat, archs4_genes)
message("Loaded and matched C2CP pathway matrix")

# Load FBM
fbm_file <- file.path(base_output_dir, "fbm")
output_file <- paste0(fbm_file, "_filtered")

archs4_fbm_filt <- FBM(
  nrow        = n_genes_thin,
  ncol        = n_samples_total,
  backingfile = output_file,
  create_bk   = FALSE
)

message("Loaded FBM with ", n_genes_thin, " genes and ", n_samples_total, " samples")

## Run 3 models with different seeds

In [ ]:
# Calculate target number of samples based on coverage
n_genes <- nrow(archs4_fbm_filt)
n_samples_target <- round(n_samples_total * coverage)
message("Target samples per run: ", n_samples_target, " (", coverage_pct, "% of ", n_samples_total, ")")

# Store results summary
results_summary <- data.frame(
  run = integer(),
  seed = integer(),
  n_samples = integer(),
  CLAMP_K = integer(),
  stringsAsFactors = FALSE
)

for (run_idx in seq_len(n_runs)) {
  current_seed <- seeds[run_idx]
  message("\n", strrep("=", 60))
  message("RUN ", run_idx, "/", n_runs, " - Seed: ", current_seed)
  message(strrep("=", 60))
  
  # Create output directory for this run
  output_dir <- file.path(base_output_dir, paste0("c2cp_coverage_rs", coverage_pct, "_seed_", run_idx))
  dir.create(output_dir, showWarnings = FALSE, recursive = TRUE)
  
  # Sample selection with current seed
  set.seed(current_seed)
  sample_idx <- sort(sample(seq_len(n_samples_total), n_samples_target))
  n_samples <- length(sample_idx)
  sample_names <- sample_names_total[sample_idx]
  
  message("Randomly selected ", n_samples, " total samples")
  
  # Save sample info
  saveRDS(list(
    run = run_idx,
    seed = current_seed,
    coverage = coverage,
    n_samples = n_samples,
    sample_idx = sample_idx,
    sample_names = sample_names,
    sampling_method = "random_sampling"
  ), file = file.path(output_dir, "subsample_info.rds"))
  
  # Create subsampled FBM
  message("Creating subsampled FBM...")
  fbm_sub_file <- file.path(output_dir, "fbm_subsampled")
  
  Y_sub <- big_copy(
    archs4_fbm_filt,
    ind.col = sample_idx,
    backingfile = fbm_sub_file
  )
  
  # SVD
  message("Computing SVD...")
  SVD_K <- round(min(n_samples - 1, n_genes - 1) / 4)
  
  if (N_CORES > 1) {
    options(bigstatsr.check.parallel.blas = FALSE)
    blas_nproc <- getOption("default.nproc.blas")
    options(default.nproc.blas = NULL)
  }
  
  svd_result <- big_randomSVD(Y_sub, k = SVD_K, ncores = N_CORES)
  
  if (N_CORES > 1) {
    options(bigstatsr.check.parallel.blas = TRUE)
    options(default.nproc.blas = blas_nproc)
  }
  
  valid_idx <- which(!is.nan(svd_result$d))
  svd_result$d <- svd_result$d[valid_idx]
  svd_result$u <- svd_result$u[, valid_idx, drop = FALSE]
  svd_result$v <- svd_result$v[, valid_idx, drop = FALSE]
  
  saveRDS(svd_result, file = file.path(output_dir, "svd.rds"))
  
  # Estimate CLAMP K using Gavish-Donoho
  eigenvalues <- sort(svd_result$d^2 / (n_samples - 1), decreasing = TRUE)
  noise_gd    <- median(eigenvalues)
  CLAMP_K     <- PCAtools::chooseGavishDonoho(
    .dim          = c(n_genes, n_samples),
    var.explained = eigenvalues,
    noise         = noise_gd
  ) * 2
  message("CLAMP K (Gavish-Donoho) = ", CLAMP_K)
  saveRDS(CLAMP_K, file = file.path(output_dir, "CLAMP_K.rds"))
  
  # CLAMPbase
  message("Running CLAMPbase...")
  baseRes <- CLAMPbase(
    Y = Y_sub,
    svdres = svd_result,
    trace = TRUE,
    clamp_k = CLAMP_K
  )
  
  baseRes$Z <- data.frame(baseRes$Z)
  rownames(baseRes$Z) <- archs4_genes
  baseRes$B <- data.frame(baseRes$B)
  colnames(baseRes$B) <- sample_names
  
  saveRDS(baseRes, file = file.path(output_dir, "CLAMPbase.rds"))
  
  model_dir <- file.path(output_dir, "CLAMPbase")
  dir.create(model_dir, showWarnings = FALSE, recursive = TRUE)
  write.csv(baseRes$B, file.path(model_dir, "B.csv"))
  write.csv(baseRes$Z, file.path(model_dir, "Z.csv"))
  
  # CLAMPfull with C2CP prior
  message("Running CLAMPfull with C2CP prior...")
  fullRes <- CLAMPfull(
    Y = Y_sub,
    svdres = svd_result,
    priorMat = C2CP_matched,
    clamp.base.result = baseRes,
    use_cpp = TRUE,
    trace = TRUE,
    multiplier      = MULTIPLIER,
    max.iter        = MAX_ITER,
    clamp_k = CLAMP_K
  )
  
  fullRes$Z <- data.frame(fullRes$Z)
  rownames(fullRes$Z) <- archs4_genes
  fullRes$B <- data.frame(fullRes$B)
  colnames(fullRes$B) <- sample_names
  fullRes$summary <- fullRes$summary %>%
    dplyr::rename(LV = LV_index) %>%
    dplyr::mutate(LV = paste0('LV', LV))
  
  saveRDS(fullRes, file = file.path(output_dir, "CLAMPfull_C2CP.rds"))
  
  model_dir <- file.path(output_dir, "CLAMPfull_C2CP")
  dir.create(model_dir, showWarnings = FALSE, recursive = TRUE)
  write.csv(fullRes$B, file.path(model_dir, "B.csv"))
  write.csv(fullRes$Z, file.path(model_dir, "Z.csv"))
  write.csv(fullRes$summary, file.path(model_dir, "summary.csv"))
  
  # Store summary
  results_summary <- rbind(results_summary, data.frame(
    run = run_idx,
    seed = current_seed,
    n_samples = n_samples,
    CLAMP_K = CLAMP_K
  ))
  
  # Clean up memory
  rm(Y_sub, svd_result, baseRes, fullRes)
  gc()
}

message("\n", strrep("=", 60))
message("All ", n_runs, " runs completed!")
message(strrep("=", 60))